# Explainability in BRB systems

A key advantage of BRB systems over black-box models is that **every intermediate quantity in the inference pipeline has a clear physical or logical meaning**. This notebook demonstrates how to extract and visualize these quantities to explain individual predictions.

For a formal treatment of BRB interpretability, see Sachan et al. (2020), *Global and local interpretability of belief rule base*. The INFRINGER method (Misitano 2020) uses this interpretability for preference modeling in multi-objective optimization.

```
# requires: pip install desdeo-brb matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from desdeo_brb import BRBModel, RuleBase
from desdeo_brb.utils import build_rule_antecedent_indices

## Setup: trained f(x) = x sin(x^2) model

We train the standard benchmark so we have a model with interesting structure to inspect.

In [ ]:
def f(x):
    return x * np.sin(x**2)

prv = [np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])]
crv = np.array([-2.5, -1.0, 1.0, 2.0, 3.0])
model = BRBModel(prv, crv, initial_rule_fn=lambda x: f(x[0]))

X_train = np.linspace(0, 3, 1000).reshape(-1, 1)
y_train = f(X_train[:, 0])
model.fit(X_train, y_train, fix_endpoints=True, n_restarts=5)

print(f"{model.rule_base.n_rules} rules, {model.rule_base.n_attributes} attribute(s)")

## Part A: Inspecting the trained rule base

The `describe_all_rules()` method gives a one-line summary of every rule in the model (note that consequent values with a belief degree of zero are omitted from the output).

In [ ]:
print(model.rule_base.describe_all_rules(
    attribute_names=["x"], consequent_name="f(x)"
))

**Reading the rules:**

- Each rule states: *IF x is [value] THEN f(x) = {distribution over consequent values} [w=weight]*
- The belief distribution tells us how the model represents the function's output at that point. For example, if Rule 3 says `{1: 0.83, 2: 0.17}` with consequent values [-2.5, -1, 1, 2, 3], the model believes the output at x=1.0 is mostly near 1.0 (83% belief) with some belief at 2.0 (17%).
- Rule weights indicate relative importance. After training, some rules become more influential than others, reflecting which parts of the input space matter most for accuracy.
- Referential values may have shifted from their initial positions (adaptive training).

## Part B: Explaining single predictions

The `explain()` method shows the full reasoning chain for a prediction.

In [ ]:
# Case 1: at a referential value (one dominant rule)
print("=== Near a referential value ===")
print(model.explain(np.array([[1.5]]), attribute_names=["x"], consequent_name="f(x)"))
print()

# Case 2: between two referential values (two rules activate)
print("=== Between referential values ===")
print(model.explain(np.array([[1.75]]), attribute_names=["x"], consequent_name="f(x)"))
print()

# Case 3: critical point where behaviour changes
print("=== Near a critical point (x~2.2) ===")
print(model.explain(np.array([[2.2]]), attribute_names=["x"], consequent_name="f(x)"))

**What the explanation tells you:**

- **Prediction** is the scalar output, a weighted average of consequent values using the combined belief distribution.
- **Top activated rules** show which rules contributed most to the prediction and their activation weights. When the input is near a referential value, one rule dominates. Between referential values, two adjacent rules share activation proportionally to the input's distance from each.
- **Combined belief distribution** is the output of the evidential reasoning algorithm: it aggregates the activated rules' belief distributions into a single output distribution.

### Visualizing rule activations

In [ ]:
X_point = np.array([[1.75]])
result = model.predict(X_point)
weights = result.activation_weights[0]
rb = model.rule_base

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Left: activation weights per rule
rule_labels = [f"R{k}\n(x={rb.precedent_referential_values[0][rb.rule_antecedent_indices[k,0]]:.2f})"
               for k in range(rb.n_rules)]
colors = ["#d62728" if w > 0.05 else "#aec7e8" for w in weights]
ax1.bar(range(len(weights)), weights, color=colors)
ax1.set_xticks(range(len(weights)))
ax1.set_xticklabels(rule_labels, fontsize=8)
ax1.set_ylabel("Activation weight")
ax1.set_title(f"Rule activations at x={X_point[0,0]}")
ax1.grid(True, alpha=0.3, axis="y")

# Right: combined belief distribution
beliefs = result.combined_belief_degrees[0]
ax2.bar(crv, beliefs, width=0.6, color="steelblue", edgecolor="black")
ax2.axvline(result.output[0], color="red", linestyle="--", linewidth=2,
            label=f"Output = {result.output[0]:.2f}")
ax2.set_xlabel("Consequent referential values")
ax2.set_ylabel("Belief degree")
ax2.set_title("Combined belief distribution")
ax2.legend()
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Part C: What-if analysis

By comparing explanations at nearby inputs, we can see how the model's reasoning shifts with small changes. This is useful for understanding sensitivity and building trust.

In [ ]:
x_base = 2.0
delta = 0.3

for x_val in [x_base - delta, x_base, x_base + delta]:
    X_q = np.array([[x_val]])
    res = model.predict(X_q)
    top = res.dominant_rules(top_k=2)[0]
    w_top = [res.activation_weights[0, k] for k in top]
    print(f"x={x_val:.1f}  pred={res.output[0]:+.3f}  true={f(x_val):+.3f}  "
          f"top rules: {[int(k) for k in top]}  weights: [{w_top[0]:.3f}, {w_top[1]:.3f}]")

print()
print("As x moves from 1.7 to 2.3, the dominant rule shifts and the")
print("activation weights redistribute smoothly between adjacent rules.")

## Pipeline leak detection: explaining a real-world scenario

The pipeline leak detection model from Chen et al. (2011) / Xu et al. (2007) is a canonical BRB application. We construct the 56-rule expert model and explain a leak scenario.

In [ ]:
# Construct the pipeline model (self-contained)
flow_diff_rv = np.array([-10.0, -4.1, -2.8, -1.79, -0.79, 0.25, 2.0, 3.0])
pressure_diff_rv = np.array([-0.01, -0.008, -0.005, 0.003, 0.0058, 0.008, 0.01])
leak_size_rv = np.array([0.0, 2.0, 4.0, 6.0, 8.0])
prv_pipe = [flow_diff_rv, pressure_diff_rv]

expert_beliefs = np.array([
    [0,0,0,0,1],[0,0,0,.3,.7],[0,0,.2,.8,0],[0,0,.8,.2,0],[.65,.35,0,0,0],[.85,.15,0,0,0],[.95,.05,0,0,0],
    [0,0,.1,.9,0],[0,0,.7,.3,0],[0,.7,.3,0,0],[0,.9,.1,0,0],[.8,.2,0,0,0],[.9,.1,0,0,0],[.99,.01,0,0,0],
    [0,0,.4,.6,0],[0,0,.8,.2,0],[0,.3,.6,.1,0],[.1,.8,.1,0,0],[.9,.1,0,0,0],[.95,.05,0,0,0],[1,0,0,0,0],
    [0,0,.5,.5,0],[0,.1,.8,.1,0],[0,.8,.2,0,0],[1,0,0,0,0],[.95,.05,0,0,0],[.98,.02,0,0,0],[1,0,0,0,0],
    [0,0,.6,.4,0],[0,.2,.6,.2,0],[.1,.6,.3,0,0],[.9,.1,0,0,0],[.95,.05,0,0,0],[.99,.01,0,0,0],[1,0,0,0,0],
    [0,0,.6,.4,0],[0,.2,.7,.1,0],[0,.7,.3,0,0],[.95,.05,0,0,0],[.99,.01,0,0,0],[1,0,0,0,0],[1,0,0,0,0],
    [0,.1,.7,.2,0],[0,.3,.6,.1,0],[.1,.7,.2,0,0],[.98,.02,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],
    [0,.1,.8,.1,0],[0,.3,.7,0,0],[.1,.8,.1,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],
])

rb_pipe = RuleBase(
    precedent_referential_values=prv_pipe,
    consequent_referential_values=leak_size_rv,
    belief_degrees=expert_beliefs,
    rule_weights=np.full(56, 1.0 / 56),
    attribute_weights=np.ones((56, 2)),
    rule_antecedent_indices=build_rule_antecedent_indices(prv_pipe),
)
pipe_model = BRBModel(prv_pipe, leak_size_rv, rule_base=rb_pipe)
print(f"Pipeline model: {pipe_model.rule_base.n_rules} rules")

In [ ]:
# Explain a potential leak scenario
X_leak = np.array([[-5.0, -0.007]])
print(pipe_model.explain(
    X_leak,
    top_k=5,
    attribute_names=["FlowDiff", "PressureDiff"],
    consequent_name="LeakSize",
))

In [ ]:
# Visualize: leak scenario vs normal operation
scenarios = {
    "Normal (FD=-0.8, PD=0.003)": np.array([[-0.8, 0.003]]),
    "Leak (FD=-5.0, PD=-0.007)": np.array([[-5.0, -0.007]]),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, X_q) in zip(axes, scenarios.items()):
    res = pipe_model.predict(X_q)
    leak_labels = ["Zero", "Small", "Med", "High", "VHigh"]
    ax.bar(leak_labels, res.combined_belief_degrees[0], color="coral", edgecolor="black")
    ax.set_ylabel("Belief degree")
    ax.set_title(f"{label}\nPredicted LeakSize = {res.output[0]:.1f}")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Part D: Comparing to black-box ML

In a neural network or gradient-boosted tree, a prediction is a single number with no built-in explanation. To understand *why* the model produced that number, you need post-hoc tools like SHAP or LIME, which approximate the model's behavior locally. These approximations are computationally expensive, not guaranteed to be faithful, and add another layer of complexity.

In a BRB system, every intermediate quantity has a direct semantic interpretation:

- **Input belief distributions** show how each input maps onto the referential value grid.
- **Activation weights** show which rules are relevant and how strongly.
- **Belief degrees** inside each rule describe the expected output as a distribution.
- **Combined belief degrees** show the aggregated output distribution after evidential reasoning.

No post-hoc approximation is needed. The explanation is exact and comes for free with every prediction. The rule base itself is human-readable (as shown by `describe_all_rules()`), and each prediction can be traced back to specific rules that an expert can validate against domain knowledge.

This intrinsic transparency makes BRB systems well-suited for domains where trust, accountability, and auditability matter: fault diagnosis, medical decision support, risk assessment, and interactive preference learning.

## Summary

This notebook demonstrated:
1. Inspecting the trained rule base with `describe_all_rules()`
2. Explaining predictions with `model.explain()` and visualizing activations
3. What-if analysis showing how activation patterns shift with input changes
4. A real-world pipeline leak detection explanation
5. How BRB explainability differs from post-hoc methods for black-box models

For the full API, see the [API reference](https://desdeo-brb.readthedocs.io/en/latest/api/models/).